# Thí nghiệm 4 — (việc 4) pipeline số nguyên bit-exact + (việc 2) baseline mạnh hơn
Chạy sau `ecg_pa_exp3` (dùng lại model 5 seed, model fold DS1 và siêu tham số đã chọn; **không huấn luyện lại model chính**).

**Việc 4.** exp3 lấy vector đặc trưng bằng fake-quant số thực với `np.round` (làm tròn *chẵn*), còn IP dùng `AP_RND` (làm tròn *nửa lên*).
Khác biệt nhỏ (≈5/10 000 nhãn) nhưng làm số liệu phần mềm lệch board ở chữ số thứ 4. Notebook này thay bằng `hw_ref.py`:
tham chiếu **số nguyên** đã kiểm chứng khớp tung bit với C-sim (49 749 nhịp) và với **dự đoán trên board** (none / offset_int / gradient_anchor, seed 3: 0 nhịp lệch).
→ Toàn bộ bảng 5 seed, Wilcoxon, ngân sách nhãn được tính lại trên pipeline này.

**Việc 2.** Baseline (siêu tham số chọn trên DS1):
- `head_scratch`: học lại lớp đầu ra **từ đầu** chỉ bằng 5 phút của bệnh nhân (không dùng trọng số toàn cục).
- `ft_full`: fine-tune **toàn bộ mạng** (54 772 tham số, số thực) bằng 5 phút đầu.
- `ft_full_anchor`: như trên + phát lại 100 nhịp neo DS1 (công bằng với `gradient_anchor`).
- `scratch_patient`: huấn luyện mạng **từ đầu** chỉ bằng 5 phút của bệnh nhân (kiểu patient-specific cổ điển).

Thời gian: ~20 phút (việc 4) + ~40–60 phút (việc 2, có GPU).

In [ ]:
!pip install -q wfdb
from google.colab import drive
drive.mount('/content/drive')

## 1. Cấu hình, tiền xử lý, model (giữ nguyên từ exp3)

In [ ]:
import os, json, time
import numpy as np

DATA_DIR = '/content/drive/MyDrive/mitdb_extracted/mit-bih-arrhythmia-database-1.0.0'   # thư mục chứa 100.dat, 100.hea, 100.atr ...
OUT_DIR  = '/content/drive/MyDrive/ecg_processed/improve_v4'                 # nơi lưu kết quả, model
os.makedirs(OUT_DIR, exist_ok=True)

FS_RAW = 360
SEED   = 42

# 4 lớp theo khuyến nghị AAMI EC57 / de Chazal (lớp Q gần như không có trong DS1/DS2
# vì các bản ghi máy tạo nhịp 102, 104, 107, 217 bị loại). Đặt USE_Q = True để giữ 5 lớp.
USE_Q = False

DS1 = [101, 106, 108, 109, 112, 114, 115, 116, 118, 119, 122, 124, 201, 203,
       205, 207, 208, 209, 215, 220, 223, 230]
DS2 = [100, 103, 105, 111, 113, 117, 121, 123, 200, 202, 210, 212, 213, 214,
       219, 221, 222, 228, 231, 232, 233, 234]

AAMI = {'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
        'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
        'V': 'V', 'E': 'V',
        'F': 'F',
        '/': 'Q', 'f': 'Q', 'Q': 'Q'}
BEAT_SYMBOLS = set(AAMI.keys())
CLASS_NAMES = ['N', 'S', 'V', 'F', 'Q'] if USE_Q else ['N', 'S', 'V', 'F']
NUM_CLASSES = len(CLASS_NAMES)
print('Số lớp:', NUM_CLASSES, CLASS_NAMES)

In [ ]:
import wfdb
from scipy.ndimage import median_filter
from scipy.signal import butter, filtfilt

def load_record(rec):
    path = os.path.join(DATA_DIR, str(rec))
    r = wfdb.rdrecord(path)
    ch = r.sig_name.index('MLII') if 'MLII' in r.sig_name else 0
    sig = r.p_signal[:, ch].astype(np.float64)
    ann = wfdb.rdann(path, 'atr')
    return sig, np.asarray(ann.sample), np.asarray(ann.symbol)

def clean_signal(sig, fs=FS_RAW):
    base = median_filter(sig, size=int(0.2 * fs) | 1, mode='nearest')
    base = median_filter(base, size=int(0.6 * fs) | 1, mode='nearest')
    x = sig - base
    b, a = butter(4, 40.0 / (fs / 2), btype='low')
    return filtfilt(b, a, x)

def extract_record(rec, pre_s, post_s, ds, filt):
    sig, samples, symbols = load_record(rec)
    if filt:
        sig = clean_signal(sig)
    fs = FS_RAW / ds
    if ds > 1:
        sig = sig[::ds]                        # đã lọc thông thấp 40 Hz nên hạ mẫu an toàn
    pre, post = int(round(pre_s * fs)), int(round(post_s * fs))

    is_beat = np.array([s in BEAT_SYMBOLS for s in symbols])
    r_all = samples[is_beat] / ds
    sym_all = symbols[is_beat]
    rr = np.diff(r_all) / fs                   # giây

    X, R, Y = [], [], []
    for i in range(1, len(r_all) - 1):
        lab = AAMI[sym_all[i]]
        if lab == 'Q' and not USE_Q:
            continue
        c = int(round(r_all[i]))
        if c - pre < 0 or c + post > len(sig):
            continue
        w = sig[c - pre:c + post]
        w = w - np.median(w)
        m = np.max(np.abs(w))
        if m < 1e-9:
            continue
        pre_rr, post_rr = rr[i - 1], rr[i]
        local = np.mean(rr[max(0, i - 10):i])
        local30 = np.mean(rr[max(0, i - 30):i])
        med300 = np.median(rr[max(0, i - 300):i])
        X.append((w / m).astype(np.float32))
        R.append([pre_rr, post_rr, local, pre_rr / local, post_rr / local, pre_rr / post_rr,
                  pre_rr / local30, post_rr / local30, pre_rr / med300, post_rr / med300])
        Y.append(CLASS_NAMES.index(lab))
    return np.array(X), np.array(R, dtype=np.float32), np.array(Y), np.full(len(Y), rec)

_cache = {}
def build_dataset(records, pre_s, post_s, ds, filt):
    key = (tuple(records), pre_s, post_s, ds, filt)
    if key not in _cache:
        parts = [extract_record(r, pre_s, post_s, ds, filt) for r in records]
        _cache[key] = tuple(np.concatenate([p[k] for p in parts]) for k in range(4))
    return _cache[key]

def describe(y, name):
    cnt = np.bincount(y, minlength=NUM_CLASSES)
    print(name, dict(zip(CLASS_NAMES, cnt.tolist())), 'tổng', len(y))

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

N_RR = 10

def build_model(win_len, conv_blocks, use_rr, dense_units=32, rr_dense=0):
    beat_in = layers.Input(shape=(win_len, 1), name='beat')
    x = beat_in
    for i, (f, k) in enumerate(conv_blocks, 1):
        x = layers.Conv1D(f, k, padding='same', name=f'conv1d_{i}')(x)
        x = layers.ReLU(name=f'relu_{i}')(x)
        x = layers.MaxPooling1D(2, name=f'pool_{i}')(x)
    x = layers.Flatten(name='flatten')(x)
    inputs = [beat_in]
    if use_rr:
        rr_in = layers.Input(shape=(CUR_NRR,), name='rr')
        inputs.append(rr_in)
        r = layers.Dense(rr_dense, activation='relu', name='rr_dense')(rr_in) if rr_dense else rr_in
        x = layers.Concatenate(name='concat')([x, r])
    x = layers.Dense(dense_units, activation='relu', name='dense_1')(x)
    out = layers.Dense(NUM_CLASSES, name='dense_out')(x)   # logit
    return models.Model(inputs, out)

CUR_NRR = 10   # số đặc trưng RR dùng cho thí nghiệm hiện tại (đặt trong vòng lặp)

def rr_normalize(R):
    """Đưa RR về khoảng ~[0, 2]: giây chia 1.5, tỉ số chia 2. Cố định, không học -> dễ làm phần cứng."""
    R = R[:, :CUR_NRR]
    return (R / np.array([1.5, 1.5, 1.5, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0], dtype=np.float32)[:CUR_NRR]).clip(0, 3.9)

In [ ]:
def class_weights_effective(y, beta=0.999, max_ratio=10.0):
    cnt = np.maximum(np.bincount(y, minlength=NUM_CLASSES), 1)
    w = (1 - beta) / (1 - np.power(beta, cnt))
    w = w / w.min()
    w = np.minimum(w, max_ratio)
    return w / np.mean(w[np.bincount(y, minlength=NUM_CLASSES) > 0])

def focal_loss(alpha, gamma=2.0):
    alpha = tf.constant(alpha, dtype=tf.float32)
    def loss(y_true, logits):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        logp = tf.nn.log_softmax(logits)
        idx = tf.stack([tf.range(tf.shape(y_true)[0]), y_true], axis=1)
        lp = tf.gather_nd(logp, idx)
        p = tf.exp(lp)
        a = tf.gather(alpha, y_true)
        return tf.reduce_mean(-a * tf.pow(1.0 - p, gamma) * lp)
    return loss

def time_stretch(x, max_frac=0.10):
    """Co giãn cả batch theo 1 hệ số ngẫu nhiên, giữ tâm (đỉnh R) cố định."""
    L = tf.shape(x)[1]
    s = tf.random.uniform([], 1.0 - max_frac, 1.0 + max_frac)
    newL = tf.cast(tf.round(tf.cast(L, tf.float32) * s), tf.int32)
    y = tf.image.resize(x[:, :, None, :], [newL, 1], method='bilinear')[:, :, 0, :]
    diff = newL - L
    y = tf.cond(diff > 0,
                lambda: y[:, diff // 2: diff // 2 + L, :],
                lambda: tf.pad(y, [[0, 0], [(-diff) // 2, -diff - (-diff) // 2], [0, 0]], mode='SYMMETRIC'))
    return tf.reshape(y, tf.shape(x))

def augment_beats(x, stretch=False):
    """x: (B, L, 1). Biên độ, nhiễu, trôi nền, dịch thời gian nhỏ (+ co giãn)."""
    if stretch:
        x = time_stretch(x)
    B = tf.shape(x)[0]; L = tf.shape(x)[1]
    scale = tf.random.uniform([B, 1, 1], 0.85, 1.15)
    noise = tf.random.normal(tf.shape(x), stddev=0.02)
    t = tf.cast(tf.range(L), tf.float32)[None, :, None] / tf.cast(L, tf.float32)
    wander = tf.random.uniform([B, 1, 1], -0.08, 0.08) * tf.sin(
        2.0 * np.pi * tf.random.uniform([B, 1, 1], 0.2, 1.0) * t + tf.random.uniform([B, 1, 1], 0, 6.28))
    shift = tf.random.uniform([], -4, 5, dtype=tf.int32)
    x = tf.roll(x, shift, axis=1)
    return tf.clip_by_value(x * scale + noise + wander, -1.98, 1.98)

def make_ds(Xb, R, y, use_rr, batch, training, stretch=False):
    feats = {'beat': Xb[..., None]}
    if use_rr:
        feats['rr'] = R
    ds = tf.data.Dataset.from_tensor_slices((feats, y))
    if training:
        ds = ds.shuffle(len(y), seed=SEED)
    ds = ds.batch(batch)
    if training:
        def aug(f, yy):
            f = dict(f); f['beat'] = augment_beats(f['beat'], stretch); return f, yy
        ds = ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
from sklearn.metrics import confusion_matrix

def aami_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    res = {}
    f1s, ses = [], []
    for c in range(NUM_CLASSES):
        tp = cm[c, c]; fn = cm[c].sum() - tp; fp = cm[:, c].sum() - tp
        se = tp / (tp + fn) if tp + fn else np.nan
        pp = tp / (tp + fp) if tp + fp else 0.0
        f1 = 2 * se * pp / (se + pp) if (se + pp) and not np.isnan(se) else 0.0
        res[CLASS_NAMES[c]] = dict(Se=se, PP=pp, F1=f1, n=int(cm[c].sum()))
        if cm[c].sum():
            f1s.append(f1); ses.append(se)
    res['acc'] = np.trace(cm) / cm.sum()
    res['macroF1'] = float(np.mean(f1s))
    res['balAcc'] = float(np.mean(ses))
    res['cm'] = cm
    return res

def print_metrics(m, title):
    print(f'--- {title} ---  acc={m["acc"]:.4f}  macroF1={m["macroF1"]:.4f}  balAcc={m["balAcc"]:.4f}')
    for c in CLASS_NAMES:
        d = m[c]
        if d['n']:
            print(f'   {c}: Se={d["Se"]:.3f}  +P={d["PP"]:.3f}  F1={d["F1"]:.3f}  (n={d["n"]})')
    print('   Confusion (hàng=thật, cột=đoán):'); print(m['cm'])

# ---------- Lượng tử hóa kiểu phần cứng (Q-format 8-bit, thang lũy thừa 2) ----------
def frac_bits_for(x, bits=8):
    m = float(np.max(np.abs(x)))
    if m < 1e-12:
        return bits - 1
    return int(max(bits - 1 - max(int(np.ceil(np.log2(m))), 0), 0))

def fq(x, f, bits=8):
    s = 2.0 ** f
    return np.clip(np.round(x * s), -2 ** (bits - 1), 2 ** (bits - 1) - 1) / s

def _conv_same(x, W, b):
    """x (n,L,C), W keras (k,C,F)."""
    k = W.shape[0]; pl = (k - 1) // 2; pr = k - 1 - pl
    xp = np.pad(x, ((0, 0), (pl, pr), (0, 0)))
    win = np.lib.stride_tricks.sliding_window_view(xp, k, axis=1)     # (n,L,C,k)
    return np.einsum('nlck,kcf->nlf', win, W, optimize=True) + b

def forward_np(model, Xb, R, conv_blocks, use_rr, qcfg=None, batch=2048, rr_dense=0):
    """Suy luận NumPy từng tầng; qcfg=None -> float, ngược lại fake-quant 8-bit như phần cứng."""
    Wd = {l.name: l.get_weights() for l in model.layers if l.get_weights()}
    Q = (lambda x, key: fq(x, qcfg[key])) if qcfg else (lambda x, key: x)
    outs = []
    for s in range(0, len(Xb), batch):
        x = Q(Xb[s:s + batch, :, None].astype(np.float64), 'input')
        for i in range(1, len(conv_blocks) + 1):
            W, b = Wd[f'conv1d_{i}']
            if qcfg:
                W, b = fq(W, qcfg[f'conv1d_{i}.kernel']), fq(b, qcfg[f'conv1d_{i}.bias'])
            x = Q(np.maximum(_conv_same(x, W, b), 0), f'relu_{i}')
            Lp = x.shape[1] // 2
            x = np.maximum(x[:, 0:2 * Lp:2], x[:, 1:2 * Lp:2])
        x = x.reshape(len(x), -1)
        if use_rr:
            r = Q(rr_normalize(R[s:s + batch]).astype(np.float64), 'rr')
            if rr_dense:
                W, b = Wd['rr_dense']
                if qcfg:
                    W, b = fq(W, qcfg['rr_dense.kernel']), fq(b, qcfg['rr_dense.bias'])
                r = Q(np.maximum(r @ W + b, 0), 'rr_dense')
            x = np.concatenate([x, r], axis=1)
        W, b = Wd['dense_1']
        if qcfg:
            W, b = fq(W, qcfg['dense_1.kernel']), fq(b, qcfg['dense_1.bias'])
        x = Q(np.maximum(x @ W + b, 0), 'dense_1')
        W, b = Wd['dense_out']
        if qcfg:
            W, b = fq(W, qcfg['dense_out.kernel']), fq(b, qcfg['dense_out.bias'])
        outs.append(x @ W + b)                    # logit giữ độ chính xác đầy đủ như phần cứng
    return np.concatenate(outs)

def calibrate(model, Xb, R, conv_blocks, use_rr, n=3000, rr_dense=0):
    """Chọn frac_bits cho trọng số + activation, hiệu chuẩn trên dữ liệu TRAIN."""
    idx = np.random.default_rng(SEED).choice(len(Xb), min(n, len(Xb)), replace=False)
    Wd = {l.name: l.get_weights() for l in model.layers if l.get_weights()}
    q = {'input': 6}
    for name, (W, b) in Wd.items():
        q[f'{name}.kernel'], q[f'{name}.bias'] = frac_bits_for(W), frac_bits_for(b)
    x = Xb[idx, :, None].astype(np.float64)
    for i in range(1, len(conv_blocks) + 1):
        W, b = Wd[f'conv1d_{i}']
        x = np.maximum(_conv_same(x, W, b), 0); q[f'relu_{i}'] = frac_bits_for(x)
        Lp = x.shape[1] // 2; x = np.maximum(x[:, 0:2 * Lp:2], x[:, 1:2 * Lp:2])
    x = x.reshape(len(x), -1)
    if use_rr:
        rr = rr_normalize(R[idx]); q['rr'] = frac_bits_for(rr)
        if rr_dense:
            W, b = Wd['rr_dense']; rr = np.maximum(rr @ W + b, 0); q['rr_dense'] = frac_bits_for(rr)
        x = np.concatenate([x, rr], axis=1)
    W, b = Wd['dense_1']; q['dense_1'] = frac_bits_for(np.maximum(x @ W + b, 0))
    return q

def tune_bias(logits, y, iters=3, grid=np.linspace(-3, 3, 61)):
    """Tìm độ lệch cộng vào logit từng lớp để tối đa macro-F1 trên VALIDATION."""
    off = np.zeros(NUM_CLASSES)
    best = aami_metrics(y, np.argmax(logits, 1))['macroF1']
    for _ in range(iters):
        present = set(np.unique(y).tolist())
        for c in [c for c in range(1, NUM_CLASSES) if c in present]:   # giữ N làm mốc; bỏ lớp vắng
            for g in grid:
                o = off.copy(); o[c] = g
                f = aami_metrics(y, np.argmax(logits + o, 1))['macroF1']
                if f > best + 1e-6:
                    best, off = f, o
    return off, best

In [ ]:
LONG = dict(pre_s=0.9, post_s=1.1, ds=2, filt=True, conv=[(16, 7), (32, 5), (32, 5)], aug=True, loss='focal')
EXPERIMENTS = [
    dict(name='F1_noRR',          **LONG, use_rr=False, n_rr=0,  rr_dense=0,  cw_cap=10, stretch=False),
    dict(name='F2_rr8_str',       **LONG, use_rr=True,  n_rr=8,  rr_dense=16, cw_cap=25, stretch=True),
    dict(name='F3_rr10_med_str',  **LONG, use_rr=True,  n_rr=10, rr_dense=16, cw_cap=25, stretch=True),
]
SEEDS = [1, 2, 3, 4, 5]
LR = 5e-4
MAX_EPOCHS, BATCH, CV_FOLDS, PATIENCE = 50, 512, 3, 10

In [ ]:
tf.keras.utils.set_random_seed(SEED)

def make_folds(recs_y, k):
    """recs_y: dict rec -> nhãn. Chia bản ghi (trừ 208) thành k nhóm cân bằng S, V."""
    order = sorted([r for r in recs_y if r != 208],
                   key=lambda r: -(np.sum(recs_y[r] == 1) * 5 + np.sum(recs_y[r] == 2)))
    folds, load = [[] for _ in range(k)], np.zeros((k, 2))
    for r in order:
        j = int(np.argmin(load[:, 0] * 5 + load[:, 1]))
        folds[j].append(r)
        load[j] += [np.sum(recs_y[r] == 1), np.sum(recs_y[r] == 2)]
    return folds

class MacroF1Checkpoint(tf.keras.callbacks.Callback):
    def __init__(self, val_inputs, yv, patience):
        super().__init__(); self.vi, self.yv, self.pat = val_inputs, yv, patience
    def on_train_begin(self, logs=None):
        self.best, self.best_ep, self.wait, self.best_w = -1, 0, 0, None
    def on_epoch_end(self, epoch, logs=None):
        lg = self.model.predict(self.vi, batch_size=2048, verbose=0)
        f = aami_metrics(self.yv, lg.argmax(1))['macroF1']
        print(f'   epoch {epoch + 1:2d}  val macroF1 = {f:.4f}')
        if f > self.best:
            self.best, self.best_ep, self.wait, self.best_w = f, epoch + 1, 0, self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.pat:
                self.model.stop_training = True
    def on_train_end(self, logs=None):
        if self.best_w is not None:
            self.model.set_weights(self.best_w)

def train_one(E, Xa, Ra, ya, epochs, val=None, seed=SEED):
    tf.keras.utils.set_random_seed(seed)
    model = build_model(Xa.shape[1], E['conv'], E['use_rr'], rr_dense=E['rr_dense'])
    cw = class_weights_effective(ya, max_ratio=E['cw_cap'])
    loss = focal_loss(cw) if E['loss'] == 'focal' else \
        tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    model.compile(tf.keras.optimizers.Adam(LR, use_ema=True, ema_momentum=0.99), loss=loss)
    ds_tr = make_ds(Xa, rr_normalize(Ra), ya, E['use_rr'], BATCH, training=E['aug'], stretch=E['stretch'])
    fit_kw = {} if E['loss'] == 'focal' else {'class_weight': dict(enumerate(cw))}
    cbs = []
    if val is not None:
        Xv, Rv, yv = val
        vi = [Xv[..., None]] + ([rr_normalize(Rv)] if E['use_rr'] else [])
        ck = MacroF1Checkpoint(vi, yv, PATIENCE); cbs.append(ck)
    model.fit(ds_tr, epochs=epochs, verbose=0, callbacks=cbs, **fit_kw)
    return model, (ck.best_ep, ck.best) if val is not None else (epochs, None)



## 2. Module `pa_adapt.py` (bản dùng trên board) và `hw_ref.py` (tham chiếu số nguyên)

In [ ]:
%%writefile pa_adapt.py
"""
pa_adapt.py - Hieu chinh theo benh nhan (patient-adaptive) cho model ECG F2.
Chi dung NumPy -> chay duoc ca tren Colab lan tren ARM Cortex-A53 cua KV260 (PYNQ).

Y tuong: giu nguyen toan bo mang, chi hieu chinh LOP DAU RA (dense_out, 32 -> 4).
Dau vao cua cac ham la vector dac trung E (dau ra dense_1 sau ReLU, 32 chieu)
va trong so toan cuc (W0: 32x4, b0: 4, b0 da cong offset tu cross-validation).

Giao thuc (AAMI EC57): 5 phut dau moi ban ghi (co nhan) de hieu chinh,
phan con lai de danh gia. Moi sieu tham so chon tren DS1 (ham tune_on_folds).
"""
import numpy as np

CLASSES = ['N', 'S', 'V', 'F']
NC = 4

# ----------------------------------------------------------------------------
# 1. Chia doan hieu chinh / danh gia
# ----------------------------------------------------------------------------
def beat_times_from_rr(rr_prev_s, rec):
    """Uoc luong thoi diem moi nhip (giay) tu RR truoc (giay), khi khong luu vi tri dinh R.
    rr_prev_s: RR_i chua chuan hoa (cot 0 cua R). Sai so ~1 nhip do nhip dau/cuoi bi bo."""
    t = np.zeros(len(rec), dtype=np.float64)
    for r in np.unique(rec):
        idx = np.where(rec == r)[0]
        t[idx] = np.cumsum(rr_prev_s[idx])
    return t

def split_calibration(t_s, rec, minutes=5.0):
    """True = thuoc doan hieu chinh (t < minutes phut, tinh tu dau moi ban ghi)."""
    cal = np.zeros(len(rec), dtype=bool)
    for r in np.unique(rec):
        idx = np.where(rec == r)[0]
        cal[idx] = (t_s[idx] - t_s[idx].min()) < minutes * 60.0
    return cal

# ----------------------------------------------------------------------------
# 2. Chi so
# ----------------------------------------------------------------------------
def confusion(y, p, nc=NC):
    cm = np.zeros((nc, nc), dtype=np.int64)
    np.add.at(cm, (y, p), 1)
    return cm

def metrics(y, p):
    cm = confusion(y, p)
    n = cm.sum()
    res = {'cm': cm, 'acc': np.trace(cm) / n}
    se, pp, f1 = {}, {}, []
    for c, name in enumerate(CLASSES):
        tp = cm[c, c]; fn = cm[c].sum() - tp; fp = cm[:, c].sum() - tp
        se[name] = tp / (tp + fn) if tp + fn else np.nan
        pp[name] = tp / (tp + fp) if tp + fp else 0.0
        if cm[c].sum():
            s, q = se[name], pp[name]
            f1.append(2 * s * q / (s + q) if s + q else 0.0)
    res['Se'], res['PP'], res['macroF1'] = se, pp, float(np.mean(f1))
    pe = (cm.sum(0) * cm.sum(1)).sum() / n ** 2
    res['kappa'] = (res['acc'] - pe) / (1 - pe)
    j = np.nansum([se['S'], se['V'], pp['S'], pp['V']])
    res['jk'] = j / 8 + res['kappa'] / 2
    return res

def fmt(m):
    return (f"acc {m['acc']:.4f} | mF1 {m['macroF1']:.4f} | k {m['kappa']:.4f} | jk {m['jk']:.4f} | "
            + ' '.join(f"{c} {m['Se'][c]:.3f}/{m['PP'][c]:.3f}" for c in CLASSES))

# ----------------------------------------------------------------------------
# 3. Luong tu hoa trong so moi theo DUNG dinh dang phan cung hien tai
# ----------------------------------------------------------------------------
def fq(x, f, bits=8):
    s = 2.0 ** f
    return np.clip(np.round(x * s), -2 ** (bits - 1), 2 ** (bits - 1) - 1) / s

def quantize_head(W, b, fW, fb_logit=14, bits_b=32):
    """W: 8-bit voi frac bits fW (giu nhu model toan cuc -> khong doi phan cung).
    b: luu 32-bit Q18.14 nhu bias da gop offset trong IP hien tai."""
    return fq(W, fW, 8), fq(b, fb_logit, bits_b)

# ----------------------------------------------------------------------------
# 4. Cac phuong phap hieu chinh (moi ham tra ve W, b moi cho 1 ban ghi)
# ----------------------------------------------------------------------------
def adapt_none(E, y, W0, b0, **kw):
    return W0.copy(), b0.copy()

def adapt_offset(E, y, W0, b0, tau=1.0, **kw):
    """Chi chinh bias: dich logit theo ti le lop cua benh nhan so voi du doan toan cuc.
    tau: he so lam tron (Dirichlet). Lop khong xuat hien -> khong bi day xuong qua manh."""
    p0 = np.bincount((E @ W0 + b0).argmax(1), minlength=NC) + tau
    pt = np.bincount(y, minlength=NC) + tau
    return W0.copy(), b0 + np.log(pt / pt.sum()) - np.log(p0 / p0.sum())

def adapt_gradient(E, y, W0, b0, lam=0.1, lr=0.5, epochs=300, cap=10.0, **kw):
    """Huan luyen lai lop dau ra bang gradient (softmax CE co trong so lop),
    phat L2 keo ve trong so toan cuc; CHI cap nhat hang cua lop co mat trong doan hieu chinh.
    Dac trung duoc chuan hoa thang do noi bo (s) de lr khong phu thuoc do lon cua E."""
    s = 1.0 / (E.std() + 1e-9)
    Es = E * s
    Wt0 = W0 / s
    Wt, b = Wt0.copy(), b0.copy()
    present = np.bincount(y, minlength=NC) > 0
    cnt = np.bincount(y, minlength=NC).astype(float)
    cw = np.where(cnt > 0, np.minimum(cnt.sum() / (NC * np.maximum(cnt, 1)), cap), 0.0)
    sw = cw[y]; sw = sw / sw.sum()
    Y = np.eye(NC)[y]
    for _ in range(epochs):
        Z = Es @ Wt + b
        Z -= Z.max(1, keepdims=True)
        P = np.exp(Z); P /= P.sum(1, keepdims=True)
        G = (P - Y) * sw[:, None]
        gW = Es.T @ G + lam * (Wt - Wt0)
        gb = G.sum(0) + lam * (b - b0)
        gW[:, ~present] = 0; gb[~present] = 0
        Wt -= lr * gW; b -= lr * gb
    return Wt * s, b

def adapt_prototype(E, y, W0, b0, alpha=0.5, min_count=5, **kw):
    """Weight imprinting: tron hang trong so cua lop c voi vector trung binh dac trung
    cua lop do (da scale ve cung do lon). Chi can cong don + chia -> cung hoa duoc (phuong an C)."""
    W = W0.copy()
    for c in range(NC):
        m = y == c
        if m.sum() >= min_count:
            mu = E[m].mean(0)
            nm = np.linalg.norm(mu)
            if nm > 1e-9:
                W[:, c] = (1 - alpha) * W0[:, c] + alpha * mu * (np.linalg.norm(W0[:, c]) / nm)
    return W, b0.copy()

METHODS = {'none': adapt_none, 'offset': adapt_offset,
           'gradient': adapt_gradient, 'prototype': adapt_prototype}

def pseudo_labels(E, W0, b0, thr=0.95):
    """Bien the KHONG nhan: lay du doan toan cuc co xac suat >= thr lam nhan gia."""
    Z = E @ W0 + b0
    P = np.exp(Z - Z.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
    keep = P.max(1) >= thr
    return keep, P.argmax(1)

# ----------------------------------------------------------------------------
# 5. Chay toan bo 1 tap (moi ban ghi hieu chinh rieng)
# ----------------------------------------------------------------------------
def run_dataset(E, y, rec, cal, W0, b0, method='none', labelled=True, fW=None,
                thr=0.95, **hp):
    """Tra ve (y_eval, pred_eval, rec_eval). Danh gia CHI tren phan ngoai doan hieu chinh,
    ke ca voi 'none', de moi phuong phap duoc so tren cung tap nhip."""
    fn = METHODS[method]
    ys, ps, rs = [], [], []
    for r in np.unique(rec):
        m = rec == r
        mc, me = m & cal, m & ~cal
        if me.sum() == 0:
            continue
        Ec, yc = E[mc], y[mc]
        if not labelled:
            keep, yhat = pseudo_labels(Ec, W0, b0, thr)
            Ec, yc = Ec[keep], yhat[keep]
        if len(yc) == 0 or method == 'none':
            W, b = W0, b0
        else:
            W, b = fn(Ec, yc, W0, b0, **hp)
        if fW is not None:
            W, b = quantize_head(W, b, fW)
        ps.append((E[me] @ W + b).argmax(1)); ys.append(y[me]); rs.append(rec[me])
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(rs)

# ----------------------------------------------------------------------------
# 6. Chon sieu tham so CHI tren DS1 (dung cac model cua tung fold cross-validation)
# ----------------------------------------------------------------------------
def tune_on_folds(folds, method, grid, score='macroF1', labelled=True, **fixed):
    """folds: list dict(E, y, rec, cal, W0, b0, fW) - moi fold la cac ban ghi VALIDATION
    cua fold do, voi E tinh bang model KHONG thay cac ban ghi nay khi train.
    grid: list dict sieu tham so. Tra ve (hp_tot_nhat, bang_ket_qua)."""
    table = []
    for hp in grid:
        Y, P = [], []
        for F in folds:
            global CURRENT_ANCHORS
            CURRENT_ANCHORS = F.get('anchors')
            y_, p_, _ = run_dataset(F['E'], F['y'], F['rec'], F['cal'], F['W0'], F['b0'],
                                    method, labelled, F.get('fW'), **fixed, **hp)
            Y.append(y_); P.append(p_)
        m = metrics(np.concatenate(Y), np.concatenate(P))
        table.append((hp, m[score], m))
    best = max(table, key=lambda t: t[1])
    return best[0], table

DEFAULT_GRIDS = {
    'offset':    [dict(tau=t) for t in (1.0, 5.0, 20.0)],
    'gradient':  [dict(lam=l, lr=0.5, epochs=300) for l in (0.01, 0.1, 1.0)],
    'prototype': [dict(alpha=a, min_count=5) for a in (0.25, 0.5, 0.75)],
    # bien the khong nhan: them nguong tin cay thr cho nhan gia
    'prototype_unl': [dict(alpha=a, min_count=5, thr=t) for a in (0.25, 0.5, 0.75) for t in (0.9, 0.99)],
}


# ============================================================================
# 7. Bo sung (thi nghiem 2): offset mot chieu, offset so nguyen, danh gia tach rieng
# ============================================================================
def _log_ratio(E, y, W0, b0, tau):
    p0 = np.bincount((E @ W0 + b0).argmax(1), minlength=NC) + tau
    pt = np.bincount(y, minlength=NC) + tau
    return np.log(pt / pt.sum()) - np.log(p0 / p0.sum())

def adapt_offset_1s(E, y, W0, b0, tau=5.0, **kw):
    """Offset MOT CHIEU: chi TANG logit cua lop CO MAT trong doan hieu chinh,
    khong bao gio day lop vang mat (hoac lop bi du doan qua nhieu) xuong."""
    present = np.bincount(y, minlength=NC) > 0
    d = np.maximum(_log_ratio(E, y, W0, b0, tau), 0.0)
    d[~present] = 0.0
    return W0.copy(), b0 + d

# --- Phien ban so nguyen cho phan cung (phuong an C) ---
LN2_Q14 = 11357            # round(ln2 * 2^14)
def mitchell_log2_q(n, fbits=8):
    """log2(n) xap xi Mitchell, dang Q.fbits: vi tri bit 1 cao nhat + phan du dich trai.
    Phan cung: bo do bit dan dau + bo dich; khong can bang tra hay phep chia."""
    n = int(n)
    k = n.bit_length() - 1
    frac = ((n - (1 << k)) << fbits) >> k if k > 0 else 0
    return (k << fbits) + frac

def offset_int_q14(n_true, n_pred, tau=5, fbits=8, one_sided=True):
    """Offset Q18.14 cho 1 lop tu hai bo dem so nguyen (n_true: nhan 5 phut dau, n_pred: du doan
    cua model goc). ln(a/b) = (log2 a - log2 b)*ln2. Chi dung cong, dich, 1 phep nhan hang so."""
    d = mitchell_log2_q(n_true + tau, fbits) - mitchell_log2_q(n_pred + tau, fbits)
    if one_sided:
        d = max(d, 0)
    return (d * LN2_Q14) >> fbits          # Q.14

def adapt_offset_1s_int(E, y, W0, b0, tau=5, fbits=8, **kw):
    """Offset mot chieu tinh HOAN TOAN bang so nguyen (dem nhip, Mitchell log2, dich bit).
    Tong so nhip cua hai phan bo bang nhau nen khong can chuan hoa."""
    n_true = np.bincount(y, minlength=NC)
    n_pred = np.bincount((E @ W0 + b0).argmax(1), minlength=NC)
    d = np.array([offset_int_q14(n_true[c], n_pred[c], int(tau), fbits) if n_true[c] > 0 else 0
                  for c in range(NC)], dtype=np.float64) / 2 ** 14
    return W0.copy(), b0 + d

METHODS.update({'offset_1s': adapt_offset_1s, 'offset_1s_int': adapt_offset_1s_int})

def run_dataset_split(E, y, rec, cal, ev, W0, b0, method='none', labelled=True, fW=None,
                      thr=0.95, **hp):
    """Nhu run_dataset nhung tach mat na hieu chinh (cal) va danh gia (ev):
    cho phep thu ngan sach 1/2/5 phut ma VAN danh gia tren cung tap nhip (sau phut thu 5)."""
    fn = METHODS[method]
    ys, ps, rs = [], [], []
    for r in np.unique(rec):
        m = rec == r
        mc, me = m & cal, m & ev
        if me.sum() == 0:
            continue
        Ec, yc = E[mc], y[mc]
        if not labelled:
            keep, yhat = pseudo_labels(Ec, W0, b0, thr)
            Ec, yc = Ec[keep], yhat[keep]
        if len(yc) == 0 or method == 'none':
            W, b = W0, b0
        else:
            W, b = fn(Ec, yc, W0, b0, **hp)
        if fW is not None:
            W, b = quantize_head(W, b, fW)
        ps.append((E[me] @ W + b).argmax(1)); ys.append(y[me]); rs.append(rec[me])
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(rs)

def split_first(t_s, rec, minutes):
    """True = thuoc 'minutes' phut dau cua moi ban ghi."""
    return split_calibration(t_s, rec, minutes)

def per_record_scores(y, p, rec, min_beats=10):
    """Chi so theo ban ghi (AAMI 'average statistics'): accuracy, balanced accuracy tren cac lop
    co mat (>= min_beats nhip), va Se cua tung lop (NaN neu lop co < min_beats nhip)."""
    out = {}
    for r in np.unique(rec):
        m = rec == r
        yy, pp = y[m], p[m]
        se = {}
        for c, name in enumerate(CLASSES):
            k = yy == c
            se[name] = (pp[k] == c).mean() if k.sum() >= min_beats else np.nan
        vals = [v for v in se.values() if not np.isnan(v)]
        out[r] = dict(acc=(yy == pp).mean(), balacc=np.mean(vals) if vals else np.nan, **{f'Se_{c}': se[c] for c in CLASSES})
    return out

DEFAULT_GRIDS.update({
    'offset_1s':     [dict(tau=t) for t in (1.0, 5.0, 20.0)],
    'offset_1s_int': [dict(tau=t, fbits=8) for t in (1, 5, 20)],
})


# ============================================================================
# 8. Thi nghiem 3: offset hai chieu so nguyen, gradient co neo, offset + gradient
# ============================================================================
CURRENT_ANCHORS = None      # (A, ya): vector dac trung lay mau tu DS1 cua CHINH model dang dung

def adapt_offset_int(E, y, W0, b0, tau=5, fbits=8, **kw):
    """Offset HAI CHIEU tinh bang so nguyen (dem nhip + log2 Mitchell + dich bit), ap cho ca 4 lop.
    Tuong duong adapt_offset (so thuc) vi tong so nhip cua hai phan bo bang nhau."""
    n_true = np.bincount(y, minlength=NC)
    n_pred = np.bincount((E @ W0 + b0).argmax(1), minlength=NC)
    d = np.array([offset_int_q14(n_true[c], n_pred[c], int(tau), fbits, one_sided=False)
                  for c in range(NC)], dtype=np.float64) / 2 ** 14
    return W0.copy(), b0 + d

def adapt_gradient_anchor(E, y, W0, b0, beta=0.3, lam=0.01, lr=0.5, epochs=300, cap=10.0, **kw):
    """Gradient co NEO (rehearsal): tron them mau dac trung cua ca 4 lop lay tu DS1 (CURRENT_ANCHORS),
    chiem ti trong beta trong ham mat mat -> lop vang mat trong 5 phut dau khong bi lan mat vung quyet dinh.
    Moi hang trong so deu duoc cap nhat (ca 4 lop deu co mat nho neo)."""
    A, ya = CURRENT_ANCHORS
    Eall = np.vstack([E, A]); yall = np.concatenate([y, ya])
    cnt = np.bincount(y, minlength=NC).astype(float)
    cw = np.where(cnt > 0, np.minimum(cnt.sum() / (NC * np.maximum(cnt, 1)), cap), 0.0)
    w_cal = cw[y]; w_cal = (1 - beta) * w_cal / w_cal.sum()
    ca = np.bincount(ya, minlength=NC).astype(float)
    w_anc = beta * (1.0 / (NC * ca[ya]))
    sw = np.concatenate([w_cal, w_anc])
    s = 1.0 / (Eall.std() + 1e-9)
    Es = Eall * s; Wt0 = W0 / s
    Wt, b = Wt0.copy(), b0.copy()
    Y = np.eye(NC)[yall]
    for _ in range(epochs):
        Z = Es @ Wt + b
        Z -= Z.max(1, keepdims=True)
        P = np.exp(Z); P /= P.sum(1, keepdims=True)
        G = (P - Y) * sw[:, None]
        Wt -= lr * (Es.T @ G + lam * (Wt - Wt0)); b -= lr * (G.sum(0) + lam * (b - b0))
    return Wt * s, b

def adapt_offset_gradient(E, y, W0, b0, tau=5.0, lam=1.0, lr=0.5, epochs=300, **kw):
    """Offset hai chieu truoc, roi gradient (L2 keo ve trong so da dich offset)."""
    W1, b1 = adapt_offset(E, y, W0, b0, tau=tau)
    return adapt_gradient(E, y, W1, b1, lam=lam, lr=lr, epochs=epochs)

METHODS.update({'offset_int': adapt_offset_int, 'gradient_anchor': adapt_gradient_anchor,
                'offset_gradient': adapt_offset_gradient})
DEFAULT_GRIDS.update({
    'offset_int':      [dict(tau=t, fbits=8) for t in (1, 5, 20)],
    'gradient_anchor': [dict(beta=b, lam=0.01, lr=0.5, epochs=300) for b in (0.1, 0.3, 0.5)],
    'offset_gradient': [dict(tau=5.0, lam=l, lr=0.5, epochs=300) for l in (0.1, 1.0, 10.0)],
})

def sample_anchor_idx(y, mask, m=25, seed=0):
    """Chon toi da m chi so moi lop trong mask (bo dem rehearsal ~ 4*m vector 32 chieu)."""
    rng = np.random.default_rng(seed)
    idx = []
    for c in range(NC):
        cand = np.where(mask & (y == c))[0]
        if len(cand):
            idx.append(rng.choice(cand, min(m, len(cand)), replace=False))
    return np.concatenate(idx)


# ============================================================================
# 9. Giao tiep IP phuong an A (ecg_f2a_top): dinh dang thanh ghi + tham chieu so nguyen
# ============================================================================
def head_to_registers(W, b, fW=7, fb=14):
    """W (32x4 so thuc), b (4) -> (w_i8[128] theo chi so o*32+i, b_q14[4] int32) dung dinh dang IP."""
    w = np.clip(np.round(np.asarray(W).T * 2 ** fW), -128, 127).astype(np.int8).reshape(-1)
    bq = np.clip(np.round(np.asarray(b) * 2 ** fb), -2 ** 31, 2 ** 31 - 1).astype(np.int64)
    return w, bq

def pack_head_words(w_i8):
    """128 int8 -> 32 tu 32 bit (4 phan tu/tu, phan tu dau o byte thap) de ghi vao vung head_w AXI-Lite."""
    return np.frombuffer(np.asarray(w_i8, dtype=np.int8).tobytes(), dtype='<u4').copy()

def logits_int(emb_q4, w_i8, b_q14):
    """Tham chieu SO NGUYEN dung nhu IP: logit_q14 = b_q14 + sum(emb_q4 * w_q7) * 2^(14-4-7)."""
    e = np.asarray(emb_q4, dtype=np.int64).reshape(-1, 32)
    w = np.asarray(w_i8, dtype=np.int64).reshape(4, 32)
    return np.asarray(b_q14, dtype=np.int64)[None, :] + (e @ w.T) * 8


In [ ]:
%%writefile hw_ref.py
"""hw_ref.py - Tham chieu SO NGUYEN bit-exact voi IP ecg_f2(a)_top (Vitis HLS).
Khac voi fake-quant cu (np.round = lam tron chan): AP_RND cua HLS lam tron NUA LEN (floor(x+0.5)),
AP_SAT bao hoa. Moi phep tinh deu tren so nguyen int64 -> khop tung bit voi C-sim/co-sim/board.
Dau vao: beats_q (N,360) int8 frac 6; rr_q (N,8) int8 frac 5. Trong so: so nguyen da luong tu (Wq) + so bit thap phan.
"""
import numpy as np

def _act(acc, f_acc, f_out, bits=8):
    """acc: so nguyen voi f_acc bit thap phan (>0 sau ReLU). ReLU -> AP_RND (nua len) -> AP_SAT."""
    acc = np.maximum(acc, 0)
    sh = f_acc - f_out
    q = (acc + (1 << (sh - 1))) >> sh if sh > 0 else acc << (-sh)
    return np.minimum(q, (1 << (bits - 1)) - 1)

def _conv(x, W, b, fx, fw, fb):
    """x (n,L,C) int; W (k,C,F) int; b (F,) int. Tra (acc, f_acc). 'same' padding nhu HLS (pad trai (k-1)//2)."""
    k = W.shape[0]; pl = (k - 1) // 2; pr = k - 1 - pl
    xp = np.pad(x, ((0, 0), (pl, pr), (0, 0)))
    win = np.lib.stride_tricks.sliding_window_view(xp, k, axis=1)        # (n,L,C,k)
    f = fx + fw
    acc = np.einsum('nlck,kcf->nlf', win, W, optimize=True) + (b.astype(np.int64) << (f - fb))
    return acc, f

def _pool(x):
    L = x.shape[1] // 2
    return np.maximum(x[:, 0:2 * L:2], x[:, 1:2 * L:2])

def quantize_params(Wd, qcfg):
    """Wd: dict ten lop -> (W, b) so thuc (Keras). Tra trong so so nguyen giong het weights.h
    (weights.h duoc sinh bang fq = np.round, nen o day cung dung np.round cho TRONG SO)."""
    P = {}
    for name in ['conv1d_1', 'conv1d_2', 'conv1d_3', 'rr_dense', 'dense_1', 'dense_out']:
        W, b = Wd[name]
        fw, fb = qcfg[f'{name}.kernel'], qcfg[f'{name}.bias']
        P[name] = (np.clip(np.round(W * 2 ** fw), -128, 127).astype(np.int64), fw,
                   np.clip(np.round(b * 2 ** fb), -128, 127).astype(np.int64), fb)
    return P

def forward_emb_int(beats_q, rr_q, P, qcfg, batch=1024):
    """-> emb (N,32) int, so bit thap phan = qcfg['dense_1'] (=4). Khop bit voi emb_out cua IP."""
    fin, frr = qcfg['input'], qcfg['rr']
    outs = []
    for s in range(0, len(beats_q), batch):
        x = beats_q[s:s + batch, :, None].astype(np.int64); fx = fin
        for i in (1, 2, 3):
            W, fw, b, fb = P[f'conv1d_{i}']
            acc, fa = _conv(x, W, b, fx, fw, fb)
            fo = qcfg[f'relu_{i}']
            x = _pool(_act(acc, fa, fo)); fx = fo
        x = x.reshape(len(x), -1)                       # thu tu (p, c) giong D1_W[o][p*32+c]
        W, fw, b, fb = P['rr_dense']
        r = rr_q[s:s + batch].astype(np.int64)
        fa = frr + fw
        r = _act(r @ W + (b << (fa - fb)), fa, qcfg['rr_dense'])
        W, fw, b, fb = P['dense_1']
        # hai phan dau vao co so bit thap phan khac nhau (pool3 vs rr_dense) -> dua ve cung thang
        f1, f2 = qcfg['relu_3'], qcfg['rr_dense']
        fa = max(f1, f2) + fw
        xx = np.concatenate([x << (max(f1, f2) - f1), r << (max(f1, f2) - f2)], axis=1)
        acc = xx @ W + (b << (fa - fb))
        outs.append(_act(acc, fa, qcfg['dense_1']))
    return np.concatenate(outs)

def head_global_int(P, off, fb_logit=14):
    """Lop dau ra toan cuc nhu IP: w int8 (frac 7), bias Q18.14 = bias 8-bit + offset CV (np.round o Q14)."""
    W, fw, b, fb = P['dense_out']
    bq = np.round((b / 2 ** fb + np.asarray(off)) * 2 ** fb_logit).astype(np.int64)
    return W.T.astype(np.int64), fw, bq                   # W.T: (4,32)

def logits_int(emb, w_oi, fw, b_q14, f_emb=4, fb_logit=14):
    """logit Q18.14 = b + sum(emb * w) << (14 - f_emb - fw)."""
    return b_q14[None, :] + (emb.astype(np.int64) @ w_oi.T) * (1 << (fb_logit - f_emb - fw))


In [ ]:
import importlib, pa_adapt as pa
importlib.reload(pa)
E = [x for x in EXPERIMENTS if x['name'] == 'F2_rr8_str'][0]
CUR_NRR = E['n_rr']
kw = dict(conv_blocks=E['conv'], use_rr=E['use_rr'], rr_dense=E['rr_dense'])
PA_DIR = os.path.join(OUT_DIR, 'pa_phaseA'); os.makedirs(PA_DIR, exist_ok=True)
print('Cấu hình:', E['name'], '| thư mục kết quả:', PA_DIR)
import hw_ref as hw
importlib.reload(hw)


## 3. Dữ liệu + thời điểm nhịp (giống exp3)

In [ ]:
def extract_times(rec, pre_s, post_s, ds, filt):
    """Chạy đúng vòng lặp của extract_record nhưng trả về thời điểm (giây) của từng nhịp được giữ."""
    sig, samples, symbols = load_record(rec)
    if filt:
        sig = clean_signal(sig)
    fs = FS_RAW / ds
    if ds > 1:
        sig = sig[::ds]
    pre, post = int(round(pre_s * fs)), int(round(post_s * fs))
    is_beat = np.array([s in BEAT_SYMBOLS for s in symbols])
    r_all = samples[is_beat] / ds; sym_all = symbols[is_beat]
    T, Y = [], []
    for i in range(1, len(r_all) - 1):
        lab = AAMI[sym_all[i]]
        if lab == 'Q' and not USE_Q:
            continue
        c = int(round(r_all[i]))
        if c - pre < 0 or c + post > len(sig):
            continue
        w = sig[c - pre:c + post]; w = w - np.median(w)
        if np.max(np.abs(w)) < 1e-9:
            continue
        T.append(r_all[i] / fs); Y.append(CLASS_NAMES.index(lab))
    return np.array(T), np.array(Y)

def build_times(records, y_ref):
    parts = [extract_times(r, E['pre_s'], E['post_s'], E['ds'], E['filt']) for r in records]
    T = np.concatenate([p[0] for p in parts]); Y = np.concatenate([p[1] for p in parts])
    assert len(Y) == len(y_ref) and np.all(Y == y_ref), 'Thời điểm không khớp thứ tự nhịp!'
    return T

X1, R1, y1, g1 = build_dataset(DS1, E['pre_s'], E['post_s'], E['ds'], E['filt'])
Xte, Rte, yte, gte = build_dataset(DS2, E['pre_s'], E['post_s'], E['ds'], E['filt'])
T1 = build_times(DS1, y1); Tte = build_times(DS2, yte)
cal_te = pa.split_calibration(Tte, gte, 5)
describe(y1, 'DS1'); describe(yte, 'DS2')
describe(yte[cal_te], 'DS2 - 5 phút đầu (hiệu chỉnh)'); describe(yte[~cal_te], 'DS2 - phần đánh giá')

In [ ]:
def forward_emb(model, Xb, R, conv_blocks, use_rr, qcfg, batch=2048, rr_dense=0):
    """Giống forward_np nhưng dừng sau dense_1 (ReLU, 8-bit) -> đúng tín hiệu phần cứng sẽ xuất ra."""
    Wd = {l.name: l.get_weights() for l in model.layers if l.get_weights()}
    Q = lambda x, key: fq(x, qcfg[key])
    outs = []
    for s in range(0, len(Xb), batch):
        x = Q(Xb[s:s + batch, :, None].astype(np.float64), 'input')
        for i in range(1, len(conv_blocks) + 1):
            W, b = Wd[f'conv1d_{i}']
            W, b = fq(W, qcfg[f'conv1d_{i}.kernel']), fq(b, qcfg[f'conv1d_{i}.bias'])
            x = Q(np.maximum(_conv_same(x, W, b), 0), f'relu_{i}')
            Lp = x.shape[1] // 2
            x = np.maximum(x[:, 0:2 * Lp:2], x[:, 1:2 * Lp:2])
        x = x.reshape(len(x), -1)
        if use_rr:
            r = Q(rr_normalize(R[s:s + batch]).astype(np.float64), 'rr')
            if rr_dense:
                W, b = Wd['rr_dense']
                W, b = fq(W, qcfg['rr_dense.kernel']), fq(b, qcfg['rr_dense.bias'])
                r = Q(np.maximum(r @ W + b, 0), 'rr_dense')
            x = np.concatenate([x, r], axis=1)
        W, b = Wd['dense_1']
        W, b = fq(W, qcfg['dense_1.kernel']), fq(b, qcfg['dense_1.bias'])
        outs.append(Q(np.maximum(x @ W + b, 0), 'dense_1'))
    return np.concatenate(outs)

def head_params(model, qcfg, off):
    """Lớp đầu ra toàn cục: W 8-bit, bias 8-bit + offset CV (phần cứng gộp vào bias Q18.14)."""
    W, b = [l for l in model.layers if l.name == 'dense_out'][0].get_weights()
    return fq(W, qcfg['dense_out.kernel']), fq(b, qcfg['dense_out.bias']) + off, qcfg['dense_out.kernel']

In [ ]:
# Lượng tử hóa đầu vào đúng như phần cứng (giống cell xuất dữ liệu cho board)
def to_i8(x, f):
    return np.clip(np.round(np.asarray(x, np.float64) * 2 ** f), -128, 127).astype(np.int8)
def q_inputs(X, R, qcfg):
    return to_i8(fq(X.astype(np.float64), qcfg['input']), qcfg['input']), \
           to_i8(fq(rr_normalize(R).astype(np.float64), qcfg['rr']), qcfg['rr'])
best_hp = json.load(open(os.path.join(PA_DIR, 'exp3_best_hp_DS1.json')))
print('Siêu tham số (chọn trên DS1 ở exp3):', best_hp)
cal5 = pa.split_first(Tte, gte, 5); ev_te = ~cal5
print('Nhịp hiệu chỉnh:', cal5.sum(), '| nhịp đánh giá:', ev_te.sum())

## 4. (Việc 4) Vector đặc trưng SỐ NGUYÊN cho 5 seed
Lưu cache `exp4_emb_int_s{seed}.npz` (~3–4 phút/seed). Emb số nguyên có 4 bit thập phân → chia 16 để đưa vào `pa_adapt`.

In [ ]:
SEED_INT = {}
for seed in [1, 2, 3, 4, 5]:
    cpath = os.path.join(PA_DIR, f'exp4_emb_int_s{seed}.npz')
    model = tf.keras.models.load_model(os.path.join(OUT_DIR, f'F2_rr8_str_s{seed}.keras'), compile=False)
    qj = json.load(open(os.path.join(OUT_DIR, f'F2_rr8_str_s{seed}_quant.json'))); qcfg = qj['qcfg']
    Wd = {l.name: l.get_weights() for l in model.layers if l.get_weights()}
    P = hw.quantize_params(Wd, qcfg)
    ai = pa.sample_anchor_idx(y1, np.ones(len(y1), bool), m=25, seed=seed)
    if os.path.exists(cpath):
        z = np.load(cpath); Ei, Ai = z['E'], z['A']
    else:
        t0 = time.time()
        bq, rq = q_inputs(Xte, Rte, qcfg); abq, arq = q_inputs(X1[ai], R1[ai], qcfg)
        Ei = hw.forward_emb_int(bq, rq, P, qcfg); Ai = hw.forward_emb_int(abq, arq, P, qcfg)
        np.savez_compressed(cpath, E=Ei, A=Ai); print(f'seed {seed}: emb số nguyên {time.time() - t0:.0f} s')
    fe = qcfg['dense_1']                                   # so bit thap phan cua emb: seed 3 = 4, seed 1 = 3 ... (KHONG co dinh 4)
    wo, fw, bq14 = hw.head_global_int(P, qj['offset'])
    W0, b0 = (wo / 2.0 ** fw).T, bq14 / 2.0 ** 14            # đúng giá trị nạp vào head_w / head_b của IP
    SEED_INT[seed] = (Ei / 2.0 ** fe, W0, b0, fw, (Ai / 2.0 ** fe, y1[ai]), Ei, wo, bq14, fe)
    # đối chiếu nhanh với fake-quant cũ của exp3
    Eold = forward_emb(model, Xte[:5000], Rte[:5000], qcfg=qcfg, **kw)
    print(f'seed {seed}: emb lệch fake-quant cũ ở {(np.abs(Eold - Ei[:5000] / 2.0 ** fe) > 1e-9).any(1).mean():.1%} nhịp (5000 nhịp đầu)')

In [ ]:
# Kiểm tra: seed 3 phải trùng TỪNG NHÃN với board (đặt f2a_board_results.npz vào PA_DIR nếu có)
BOARD = os.path.join(PA_DIR, 'f2a_board_results.npz')
E3, W0, b0, fw, anc, Ei3, wo3, bq3, fe3 = SEED_INT[3]
lg = hw.logits_int(Ei3, wo3, fw, bq3, f_emb=fe3)
print('Không hiệu chỉnh, logit số nguyên vs fake-quant float trùng nhãn:', (lg.argmax(1) == (E3 @ W0 + b0).argmax(1)).mean())
for mth in ['none', 'offset_int', 'gradient_anchor']:
    pa.CURRENT_ANCHORS = anc
    y_, p_, _ = pa.run_dataset_split(E3, yte, gte, cal5, ev_te, W0, b0, mth, True, fw, **best_hp.get(mth, {}))
    msg = pa.fmt(pa.metrics(y_, p_))
    if os.path.exists(BOARD):
        bp = np.load(BOARD)[f'pred_{mth}'][ev_te]
        msg += f' | lệch board: {(bp != p_).sum()} nhịp'
    print(f'{mth:16s}', msg)
print('Board (100 MHz): none acc 0.9387 jk 0.6501 | offset_int 0.9724 / 0.8668 | gradient_anchor 0.9762 / 0.8939')

## 5. (Việc 4) Bảng chính 5 seed, Wilcoxon, ngân sách nhãn — trên pipeline số nguyên

In [ ]:
from scipy.stats import wilcoxon
import pandas as pd
METHODS_MAIN = ['none', 'offset', 'offset_int', 'gradient', 'gradient_anchor', 'offset_gradient']
rows, rec_scores = [], {}
for seed, (E_, W0, b0, fw, anc, *_rest) in SEED_INT.items():
    pa.CURRENT_ANCHORS = anc
    for mth in METHODS_MAIN:
        y_, p_, r_ = pa.run_dataset_split(E_, yte, gte, cal5, ev_te, W0, b0, mth, True, fw, **best_hp.get(mth, {}))
        m = pa.metrics(y_, p_)
        row = dict(seed=seed, method=mth, acc=m['acc'], macroF1=m['macroF1'], kappa=m['kappa'], jk=m['jk'])
        for c in pa.CLASSES:
            row[f'Se_{c}'] = m['Se'][c]; row[f'PP_{c}'] = m['PP'][c]
        rows.append(row); rec_scores[(seed, mth)] = pa.per_record_scores(y_, p_, r_)
res = pd.DataFrame(rows); res.to_csv(os.path.join(PA_DIR, 'exp4_DS2_gross_int.csv'), index=False)
cols = ['acc', 'macroF1', 'kappa', 'jk', 'Se_S', 'PP_S', 'Se_V', 'PP_V', 'Se_F', 'PP_F']
gross = res.groupby('method')[cols].agg(lambda v: f'{v.mean():.3f} ± {v.std():.3f}').loc[METHODS_MAIN]
display(gross); gross.to_csv(os.path.join(PA_DIR, 'exp4_DS2_gross_int_summary.csv'))

recs = sorted(DS2)
def rec_table(mth, key, scores=rec_scores, seeds=SEED_INT):
    return np.array([[scores[(s, mth)][r][key] for r in recs] for s in seeds]).mean(0)
base = rec_table('none', 'balacc'); w_rows = []
for mth in METHODS_MAIN[1:]:
    v = rec_table(mth, 'balacc'); d = v - base
    w_rows.append(dict(method=mth, better=int((d > 1e-9).sum()), worse=int((d < -1e-9).sum()),
                       median_gain=np.median(d), p_value=wilcoxon(v, base).pvalue))
wt = pd.DataFrame(w_rows).set_index('method'); display(wt.round(5)); wt.to_csv(os.path.join(PA_DIR, 'exp4_wilcoxon_int.csv'))

bud_rows = []
for B in [1, 2, 5]:
    calB = pa.split_first(Tte, gte, B)
    for seed, (E_, W0, b0, fw, anc, *_rest) in SEED_INT.items():
        pa.CURRENT_ANCHORS = anc
        for mth in ['offset_int', 'gradient_anchor', 'offset_gradient']:
            y_, p_, _ = pa.run_dataset_split(E_, yte, gte, calB, ev_te, W0, b0, mth, True, fw, **best_hp[mth])
            m = pa.metrics(y_, p_)
            bud_rows.append(dict(budget_min=B, seed=seed, method=mth, macroF1=m['macroF1'], jk=m['jk'],
                                 Se_S=m['Se']['S'], PP_S=m['PP']['S'], Se_F=m['Se']['F']))
bud = pd.DataFrame(bud_rows); bud.to_csv(os.path.join(PA_DIR, 'exp4_budget_int.csv'), index=False)
display(bud.groupby(['method', 'budget_min'])[['macroF1', 'jk', 'Se_S', 'PP_S', 'Se_F']].agg(lambda v: f'{v.mean():.3f} ± {v.std():.3f}'))

## 6. (Việc 2) Baseline: học lại lớp đầu ra từ đầu (`head_scratch`)
Cùng bộ tối ưu với `gradient` nhưng khởi tạo trọng số = 0 và **không** kéo về trọng số toàn cục (λ = 0). Số vòng chọn trên DS1.

In [ ]:
def adapt_head_scratch(E, y, W0, b0, lr=0.5, epochs=300, cap=10.0, **kw):
    cnt = np.bincount(y, minlength=pa.NC).astype(float)
    cw = np.where(cnt > 0, np.minimum(cnt.sum() / (pa.NC * np.maximum(cnt, 1)), cap), 0.0)
    sw = cw[y] / cw[y].sum()
    s = 1.0 / (E.std() + 1e-9); Es = E * s
    W = np.zeros_like(W0); b = np.zeros_like(b0)
    # lớp vắng mặt trong 5 phút đầu: không thể học -> logit rất âm (đúng bản chất "học từ đầu")
    b[cnt == 0] = -1e3
    Y = np.eye(pa.NC)[y]
    for _ in range(epochs):
        Z = Es @ W + b; Z -= Z.max(1, keepdims=True)
        Pp = np.exp(Z); Pp /= Pp.sum(1, keepdims=True)
        G = (Pp - Y) * sw[:, None]
        W -= lr * Es.T @ G; b -= lr * G.sum(0) * (cnt > 0)
    return W * s, b
pa.METHODS['head_scratch'] = adapt_head_scratch
pa.DEFAULT_GRIDS['head_scratch'] = [dict(lr=0.5, epochs=e) for e in (50, 150, 300)]

## 7. (Việc 2) Dữ liệu DS1-CV cho chọn siêu tham số baseline
Dùng lại model fold đã lưu ở exp3 (`fold_models`), cùng cách chia 3 fold.

In [ ]:
FOLD_DIR = os.path.join(PA_DIR, 'fold_models')
folds = make_folds({r: y1[g1 == r] for r in DS1}, CV_FOLDS)
SEEDS_TUNE = [1, 2, 3, 4, 5]          # đặt [1] nếu muốn chọn nhanh hơn
fold_info = []
for seed in SEEDS_TUNE:
    for k, vrecs in enumerate(folds):
        vmask = np.isin(g1, vrecs)
        model = tf.keras.models.load_model(os.path.join(FOLD_DIR, f'F2_fold{k}_s{seed}.keras'), compile=False)
        fold_info.append(dict(seed=seed, k=k, vmask=vmask, path=os.path.join(FOLD_DIR, f'F2_fold{k}_s{seed}.keras')))
print(len(fold_info), 'model fold')

# head_scratch: chọn số vòng trên DS1 bằng hàm tune_on_folds có sẵn (dùng lại fold_sets của exp3 nếu cần)
fold_sets = []
off_final = np.array(json.load(open(os.path.join(OUT_DIR, 'F2_rr8_str_s1_quant.json')))['offset'])
for F in fold_info:
    model = tf.keras.models.load_model(F['path'], compile=False); vmask = F['vmask']
    qcfg = calibrate(model, X1[~vmask], R1[~vmask], **kw)
    Ev = forward_emb(model, X1[vmask], R1[vmask], qcfg=qcfg, **kw)
    W0, b0, fW = head_params(model, qcfg, off_final)
    fold_sets.append(dict(E=Ev, y=y1[vmask], rec=g1[vmask], anchors=None,
                          cal=pa.split_calibration(T1[vmask], g1[vmask], 5), W0=W0, b0=b0, fW=fW,
                          seed=F['seed'], fold=F['k'] + 1))
hp_hs, tab = pa.tune_on_folds(fold_sets, 'head_scratch', pa.DEFAULT_GRIDS['head_scratch'], score='macroF1')
best_hp['head_scratch'] = hp_hs
print('head_scratch chọn trên DS1:', hp_hs)

## 8. (Việc 2) Fine-tune toàn mạng / huấn luyện từ đầu theo bệnh nhân
Mỗi bản ghi: sao chép model, huấn luyện trên 5 phút đầu (focal loss, trọng số lớp như huấn luyện gốc, **không** augment), đánh giá phần sau phút 5 bằng model **số thực** (cận trên: chưa tính mất mát do lượng tử).
Siêu tham số (lr, epoch) chọn trên DS1-CV theo macro-F1.

In [ ]:
def ft_record(base_model, Xc, Rc, yc, lr, epochs, anchors=None, scratch=False, seed=0):
    tf.keras.utils.set_random_seed(seed)
    if scratch:
        m = build_model(Xc.shape[1], E['conv'], E['use_rr'], rr_dense=E['rr_dense'])
    else:
        m = tf.keras.models.clone_model(base_model); m.set_weights(base_model.get_weights())
    if anchors is not None:
        Xa, Ra, ya = anchors
        Xc, Rc, yc = np.concatenate([Xc, Xa]), np.concatenate([Rc, Ra]), np.concatenate([yc, ya])
    cw = class_weights_effective(yc, max_ratio=E['cw_cap'])
    m.compile(tf.keras.optimizers.Adam(lr), loss=focal_loss(cw))
    t0 = time.time()
    m.fit([Xc[..., None], rr_normalize(Rc)], yc, epochs=epochs, batch_size=64, verbose=0)
    return m, time.time() - t0

def eval_ft(base_model, Xs, Rs, ys, recs_, cal, ev, lr, epochs, mode, anchors=None, seed=0, off=None):
    ys_, ps_, rs_, times = [], [], [], []
    for r in np.unique(recs_):
        mc, me = (recs_ == r) & cal, (recs_ == r) & ev
        if me.sum() == 0 or mc.sum() == 0:
            continue
        m, dt = ft_record(base_model, Xs[mc], Rs[mc], ys[mc], lr, epochs,
                          anchors=anchors if mode == 'ft_full_anchor' else None,
                          scratch=(mode == 'scratch_patient'), seed=seed)
        lg = m.predict([Xs[me][..., None], rr_normalize(Rs[me])], batch_size=2048, verbose=0)
        ps_.append(lg.argmax(1)); ys_.append(ys[me]); rs_.append(recs_[me]); times.append(dt)
        tf.keras.backend.clear_session()
    return np.concatenate(ys_), np.concatenate(ps_), np.concatenate(rs_), np.array(times)

FT_GRID = {'ft_full': [dict(lr=l, epochs=e) for l in (1e-4, 5e-4) for e in (10, 30)],
           'ft_full_anchor': [dict(lr=l, epochs=e) for l in (1e-4, 5e-4) for e in (10, 30)],
           'scratch_patient': [dict(lr=1e-3, epochs=e) for e in (30, 60)]}
TUNE_SEEDS_FT = [1]                    # để rút ngắn thời gian: chọn siêu tham số trên các fold của seed 1
ft_hp, ft_tune_rows = {}, []
for mode, grid in FT_GRID.items():
    best = (-1, None)
    for hp in grid:
        scores = []
        for F in [f for f in fold_info if f['seed'] in TUNE_SEEDS_FT]:
            base = tf.keras.models.load_model(F['path'], compile=False); vmask = F['vmask']
            ai = pa.sample_anchor_idx(y1, ~vmask, m=25, seed=F['seed'])
            calv = pa.split_calibration(T1[vmask], g1[vmask], 5)
            y_, p_, _, _ = eval_ft(base, X1[vmask], R1[vmask], y1[vmask], g1[vmask], calv, ~calv,
                                   mode=mode, anchors=(X1[ai], R1[ai], y1[ai]), seed=F['seed'], **hp)
            scores.append(pa.metrics(y_, p_)['macroF1'])
        sc = float(np.mean(scores)); ft_tune_rows.append(dict(method=mode, hp=json.dumps(hp), macroF1=sc))
        print(f'DS1-CV {mode:16s} {hp} -> macroF1 {sc:.4f}')
        if sc > best[0]: best = (sc, hp)
    ft_hp[mode] = best[1]
pd.DataFrame(ft_tune_rows).to_csv(os.path.join(PA_DIR, 'exp4_ft_tuning_DS1.csv'), index=False)
json.dump(dict(**best_hp, **ft_hp), open(os.path.join(PA_DIR, 'exp4_best_hp_DS1.json'), 'w'), indent=1)
print('Chọn:', ft_hp)

## 9. (Việc 2) Đánh giá baseline trên DS2 (5 seed), cùng tập nhịp với bảng chính

In [ ]:
bl_rows, bl_times = [], []
for seed in [1, 2, 3, 4, 5]:
    E_, W0, b0, fw, anc, *_rest = SEED_INT[seed]
    # head_scratch: trên emb số nguyên, lượng tử head như phần cứng
    y_, p_, r_ = pa.run_dataset_split(E_, yte, gte, cal5, ev_te, W0, b0, 'head_scratch', True, fw, **best_hp['head_scratch'])
    m = pa.metrics(y_, p_); rec_scores[(seed, 'head_scratch')] = pa.per_record_scores(y_, p_, r_)
    bl_rows.append(dict(seed=seed, method='head_scratch', acc=m['acc'], macroF1=m['macroF1'], kappa=m['kappa'], jk=m['jk'],
                        **{f'Se_{c}': m['Se'][c] for c in pa.CLASSES}, **{f'PP_{c}': m['PP'][c] for c in pa.CLASSES}))
    base = tf.keras.models.load_model(os.path.join(OUT_DIR, f'F2_rr8_str_s{seed}.keras'), compile=False)
    ai = pa.sample_anchor_idx(y1, np.ones(len(y1), bool), m=25, seed=seed)
    for mode in ['ft_full', 'ft_full_anchor', 'scratch_patient']:
        y_, p_, r_, tt = eval_ft(base, Xte, Rte, yte, gte, cal5, ev_te, mode=mode,
                                 anchors=(X1[ai], R1[ai], y1[ai]), seed=seed, **ft_hp[mode])
        m = pa.metrics(y_, p_); rec_scores[(seed, mode)] = pa.per_record_scores(y_, p_, r_)
        bl_rows.append(dict(seed=seed, method=mode, acc=m['acc'], macroF1=m['macroF1'], kappa=m['kappa'], jk=m['jk'],
                            **{f'Se_{c}': m['Se'][c] for c in pa.CLASSES}, **{f'PP_{c}': m['PP'][c] for c in pa.CLASSES}))
        bl_times.append(dict(seed=seed, method=mode, t_mean_s=tt.mean(), t_max_s=tt.max()))
        print(f'seed {seed} {mode:16s}', pa.fmt(m), f'| {tt.mean():.1f} s/bản ghi (Colab)')
bl = pd.DataFrame(bl_rows); bl.to_csv(os.path.join(PA_DIR, 'exp4_baselines_DS2.csv'), index=False)
pd.DataFrame(bl_times).to_csv(os.path.join(PA_DIR, 'exp4_baseline_times.csv'), index=False)

## 10. Bảng so sánh cuối: phương pháp đề xuất vs baseline (5 seed) + chi phí tính toán

In [ ]:
allres = pd.concat([res, bl], ignore_index=True)
ORDER = ['none', 'head_scratch', 'scratch_patient', 'ft_full', 'ft_full_anchor', 'offset_int', 'gradient_anchor']
summ = allres.groupby('method')[cols].agg(lambda v: f'{v.mean():.3f} ± {v.std():.3f}').loc[ORDER]

# Chi phí cập nhật cho 1 bản ghi (ước lượng MAC): n = số nhịp hiệu chỉnh trung bình
n_cal = np.mean([((gte == r) & cal5).sum() for r in DS2]); n_anc = 100
MAC_FWD = 1_008_768; HEAD = 32 * 4
def ep(m): return (best_hp.get(m) or ft_hp.get(m) or {}).get('epochs', 0)
cost = {'none': (0, 0),
        'offset_int': (0, 0),                                          # chỉ đếm nhịp + log2 Mitchell (không MAC)
        'head_scratch': (HEAD + 4, 3 * HEAD * n_cal * ep('head_scratch')),
        'gradient_anchor': (HEAD + 4, 3 * HEAD * (n_cal + n_anc) * ep('gradient_anchor')),
        'ft_full': (54_772, 3 * MAC_FWD * n_cal * ep('ft_full')),
        'ft_full_anchor': (54_772, 3 * MAC_FWD * (n_cal + n_anc) * ep('ft_full_anchor')),
        'scratch_patient': (54_772, 3 * MAC_FWD * n_cal * ep('scratch_patient'))}
summ['tham số cập nhật'] = [cost[m][0] for m in ORDER]
summ['MAC cập nhật/bản ghi'] = [f'{cost[m][1]:.2e}' for m in ORDER]
summ['đổi phần cứng?'] = ['—', 'không', 'có (nạp lại toàn bộ trọng số)', 'có (nạp lại toàn bộ trọng số)',
                          'có (nạp lại toàn bộ trọng số)', 'không (chỉ head_b)', 'không (head_w, head_b)']
display(summ); summ.to_csv(os.path.join(PA_DIR, 'exp4_final_comparison.csv'))

base_b = rec_table('none', 'balacc'); w2 = []
for mth in ORDER[1:]:
    v = rec_table(mth, 'balacc'); d = v - base_b
    w2.append(dict(method=mth, better=int((d > 1e-9).sum()), worse=int((d < -1e-9).sum()),
                   median_gain=np.median(d), p_vs_none=wilcoxon(v, base_b).pvalue))
# phương pháp đề xuất so với baseline mạnh nhất (fine-tune toàn mạng có neo)
for prop in ['offset_int', 'gradient_anchor']:
    v, b2 = rec_table(prop, 'balacc'), rec_table('ft_full_anchor', 'balacc')
    print(f'{prop} vs ft_full_anchor: tốt hơn {(v > b2 + 1e-9).sum()} / kém hơn {(v < b2 - 1e-9).sum()} bản ghi, p = {wilcoxon(v, b2).pvalue:.4g}')
display(pd.DataFrame(w2).set_index('method').round(5))
print('Đã lưu exp4_* vào', PA_DIR)